# Model Context Protocol

Assistant version 8: Connect inventory MCP tools to create_agent.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra agents --group authoring`, then `uv run --extra agents --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Official mcp SDK MCPServer; typed inventory tools; memory and stdio lifecycle

Predict the output before running. The function definitions below match `examples/01_official_server.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Discover a real official-SDK inventory server over an in-process protocol connection.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/examples/01_official_server.py --offline
Expected: One reviewed inventory_stock tool and the BW-001 record.
Cost: No model calls; local protocol only.
Requires: agents
"""

import asyncio

from common.errors import run_cli
from common.lesson import lesson_settings
from common.mcp import call_inventory, inventory_adapter
from common.pretty import print_text


def main() -> None:
    lesson_settings()

    async def exchange() -> None:
        # KEY IDEA: this is an actual MCP handshake and call, with no external socket.
        async with inventory_adapter("memory") as adapter:
            print_text(str([t.name for t in await adapter.list_tools()]))
            print_text(str(await call_inventory(adapter, "BW-001")))

    asyncio.run(exchange())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Current MCPAdapter list_tools; stdio; compare replaced MultiServerMCPClient in migration notes

Predict the output before running. The function definitions below match `examples/02_stdio_client.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Discover and call inventory tools over a managed child-process pipe.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/examples/02_stdio_client.py --offline
Expected: The official server responds over stdio and exits when the connection closes.
Cost: No model calls; local child process only.
Requires: agents
"""

import asyncio

from common.errors import run_cli
from common.lesson import lesson_settings
from common.mcp import call_inventory, inventory_adapter
from common.pretty import print_text


def main() -> None:
    lesson_settings()

    async def exchange() -> None:
        # KEY IDEA: the reviewed executable and arguments are fixed, never model-generated.
        async with inventory_adapter("stdio") as adapter:
            print_text(str(await call_inventory(adapter, "BW-001")))

    asyncio.run(exchange())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Streamable HTTP through actual in-process ASGI handlers; endpoint validation and session lifetime

Predict the output before running. The function definitions below match `examples/03_http_client.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Exercise Streamable HTTP with an ASGI transport and explicit endpoint validation.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/examples/03_http_client.py --offline
Expected: HTTP protocol discovery and call succeed without binding a network port.
Cost: No model calls; real HTTP handlers, no network sockets.
Requires: agents
"""

import asyncio

from common.errors import run_cli
from common.lesson import lesson_settings
from common.mcp import call_inventory, inventory_adapter, validate_endpoint
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    print_text(validate_endpoint("http://127.0.0.1:8765/mcp"))

    async def exchange() -> None:
        # KEY IDEA: ASGI drives the actual HTTP handlers in process; it is not a remote-server test.
        async with inventory_adapter("http") as adapter:
            print_text(str(await call_inventory(adapter, "BW-001")))

    asyncio.run(exchange())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Pass discovered tools into create_agent; managed connection cleanup

Predict the output before running. The function definitions below match `examples/04_agent_connection.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Connect reviewed discovered MCP tools to create_agent inside a managed session.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/examples/04_agent_connection.py --offline
Expected: The agent calls inventory_stock and receives validated inventory data.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

import asyncio

from langchain.agents import create_agent
from langchain.tools import tool

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.mcp import call_inventory, checked_tools, inventory_adapter
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()

    async def exchange() -> None:
        async with inventory_adapter("memory") as adapter:
            checked_tools(await adapter.list_tools())

            @tool
            async def inventory_stock(sku: str) -> dict:
                """Read public inventory through the reviewed MCP server and validate the result."""
                return await call_inventory(adapter, sku)

            # KEY IDEA: a small wrapper validates untrusted server results before agent consumption.
            agent = create_agent(
                get_chat_model(
                    settings=settings,
                    responses=[
                        tool_request("inventory_stock", {"sku": "BW-001"}),
                        "Stock checked; no reservation created.",
                    ],
                ),
                [inventory_stock],
            )
            result = await agent.ainvoke(
                {"messages": [{"role": "user", "content": "Check BW-001 stock."}]},
                {"recursion_limit": 8},
            )
            print_text(str([(m.type, m.content) for m in result["messages"]]))

    asyncio.run(exchange())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Trust review; server/tool allowlist; credentials, side effects and result validation

Predict the output before running. The function definitions below match `examples/05_trust_boundary.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reject unknown endpoints, tool sets and malformed inventory results.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/examples/05_trust_boundary.py --offline
Expected: Three explicit validation failures; trusted annotations are not assumed.
Cost: No model calls or external connections.
Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.mcp import InventoryResult, checked_tools, validate_endpoint
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    # KEY IDEA: protocol compatibility does not grant execution permission or establish truth.
    for operation in [
        lambda: validate_endpoint("https://unreviewed.example.test/mcp"),
        lambda: checked_tools([]),
        lambda: InventoryResult(sku="BW-001", stock=-1, name="Ridge Tent"),
    ]:
        try:
            operation()
        except ValueError as error:
            print_text(f"Rejected: {type(error).__name__}: {error}")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Connect inventory MCP tools to create_agent

Predict the output before running. The function definitions below match `use_case/inventory_agent.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Ask a stock question through a local MCP subprocess and a bounded agent.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/13_mcp/use_case/inventory_agent.py --offline
Expected: Reviewed server stock reaches the agent; the session and child process close.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

import asyncio

from langchain.agents import create_agent
from langchain.tools import tool

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.mcp import call_inventory, checked_tools, inventory_adapter
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()

    async def exchange() -> None:
        async with inventory_adapter("stdio") as adapter:
            checked_tools(await adapter.list_tools())

            @tool
            async def inventory_stock(sku: str) -> dict:
                """Read validated stock from the reviewed local inventory service."""
                return await call_inventory(adapter, sku)

            # KEY IDEA: keep the adapter lifetime outside the agent's entire asynchronous run.
            agent = create_agent(
                get_chat_model(
                    settings=settings,
                    responses=[
                        tool_request("inventory_stock", {"sku": "BW-001"}),
                        "Inventory checked; stock is not reserved.",
                    ],
                ),
                [inventory_stock],
                system_prompt="Use inventory evidence; never claim a purchase or reservation.",
            )
            result = await agent.ainvoke(
                {"messages": [{"role": "user", "content": "Is BW-001 in stock?"}]},
                {"recursion_limit": 8},
            )
            for message in result["messages"]:
                print_text(f"{message.type}: {message.content}")

    asyncio.run(exchange())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Official SDK server with a finite default demonstration

Predict the output before running. The function definitions below match `use_case/inventory_server.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Official SDK server; --stdio serves until its client closes the pipe.

Run: uv run --extra agents python modules/13_mcp/use_case/inventory_server.py
Expected: the finite default prints a checked inventory record; --stdio serves MCP.
Cost: no model calls; synthetic read-only inventory only.
Requires: agents
"""

import argparse
import asyncio

from common.mcp import inventory, inventory_server
from common.pretty import print_text
from common.testing import offline_environment


async def serve_stdio() -> None:
    # KEY IDEA: stdout belongs to the protocol; never print diagnostic prose to it.
    await inventory_server().run_stdio_async()


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--offline", action="store_true")
    parser.add_argument("--stdio", action="store_true")
    args = parser.parse_args()
    with offline_environment():
        if args.stdio:
            asyncio.run(serve_stdio())
        else:
            print_text(str(inventory("BW-001")))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/13_mcp/tests` for the supplied solution checks.